# Interactive METS-R + Simu5G V2X Attack Example

This notebook is a Simu5G-backed replacement sketch for Attack 1 in `security_examples.ipynb`. Instead of reading and replaying BSMs through Kafka, it keeps METS-R as the interactive simulator, sends vehicle mobility and BSM payloads to the OMNeT++ bridge with `VeinsClient.sync_tick()`, and lets the real Simu5G Uu backend decide delivery, loss, and latency.

Start the Simu5G bridge in a separate terminal before running the notebook:

```bash
cd veins_bridge/omnetpp
bash ./build_sim5g.sh
bash ./run_sim5g_uu.sh
```

The notebook can connect to an already-running METS-R interactive instance, or launch the METS-R Docker run. The results are pretty much the same as in `security_examples.ipynb`, except this one also provide the communication latency.

In [ ]:
from pathlib import Path
import copy
import json
import math
import os
import sys
from collections import deque

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

if os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from clients.METSRClient import METSRClient
from clients.VeinsClient import VeinsClient, build_mobility_records
from tutorials.v2x_veins_cosim_example import communication_records_from_result
from utils.util import read_run_config, prepare_sim_dirs, run_simulation_in_docker

print("Working directory:", os.getcwd())

## Experiment knobs

The defaults mirror the original BSM replay attack's CARLA Town 07 config and private connected vehicles. Keep `REQUIRE_SIMU5G_UU = True` when you want to guarantee that this is not falling back to the abstract bridge.

In [ ]:
RUN_CONFIG_PATH = f"{os.getcwd()}/configs/run_cosim_CARLAT7.json"

METSR_HOST = "localhost"
METSR_PORT = 4000
VEINS_HOST = "127.0.0.1"
VEINS_PORT = 9099
METSR_TRANSFORM_COORDS = True  # Use metric x/y coordinates for Euclidean distance and Simu5G mobility.

REPLAY_SOURCE_VID = 0
REPLAY_SOURCE_RADIO_ID = 1000  # Simu5G/bridge vehicle ids must be nonzero.
EGO_VID = 1
PRIVATE_VEHICLE = True
ORIGIN_ROAD = "-20"
DESTINATION_ROAD = "-20"

# Match Attack 1 in security_examples.ipynb: 3000 ticks to record,
# then 6000 ticks for each normal/attacked run at the config step size.
RECORD_TICKS = 3000
RUN_TICKS = 6000
NORMAL_TICKS = RUN_TICKS
ATTACK_TICKS = RUN_TICKS
REPLAY_OFFSET_TICKS = 10
PAYLOAD_BYTES = 320
TABLE_ROWS = 30

TARGET_VELOCITY_MPS = 10.0
MAX_ACCELERATION_MPS2 = 2.0
KP_SPEED = 0.5
MIN_GAP_M = 5.0
SLOW_SOURCE_TARGET_SPEED_MPS = 1.0
SLOW_SOURCE_BRAKE_MPS2 = -0.5

In [ ]:
def bridge_field(payload, key, default=None):
    if not isinstance(payload, dict):
        return default
    data = payload.get("data")
    if isinstance(data, dict) and key in data:
        return data[key]
    return payload.get(key, default)
def vehicle_id(record):
    return record.get("ID", record.get("vehicle_id", record.get("vid")))
def vehicle_has_road(record):
    return bool(record) and any(record.get(key) is not None for key in ("road", "road_id", "roadID"))

# Normalization function to convert METS-R vehicle records to a consistent format for analysis and replay.
def normalize_vehicle(record, private_veh=PRIVATE_VEHICLE, role="vehicle"):
    return {
        "ID": vehicle_id(record),
        "role": role,
        "x": record.get("x"),
        "y": record.get("y"),
        "z": record.get("z", 0.0),
        "speed": record.get("speed", record.get("speed_mps", 0.0)),
        "bearing": record.get("bearing", record.get("heading_deg", 0.0)),
        "acc": record.get("acc", record.get("acceleration_mps2")),
        "road": record.get("road", record.get("road_id")),
        "lane": record.get("lane", record.get("lane_id")),
        "state": record.get("state"),
        "v_type": record.get("v_type", record.get("vehicle_type")),
        "private_veh": bool(private_veh),
        "sensor_type": "cv2x",
        "map_name": getattr(config, "carla_map", "Town07"),
    }


def query_vehicle_state(metsr, vid, private_veh=PRIVATE_VEHICLE):
    records = metsr.query_vehicle(
        vid,
        private_veh=private_veh,
        transform_coords=METSR_TRANSFORM_COORDS,
    ).get("DATA", [])
    if records:
        record = dict(records[0])
        record["metsr_internal_vehicle_id"] = record.get("ID")
        record["ID"] = vid
        return record
    raise RuntimeError(f"Vehicle {vid} was not returned by METS-R.")


def wait_until_on_road(metsr, vid, private_veh=PRIVATE_VEHICLE, max_ticks=180):
    for _ in range(max_ticks):
        record = query_vehicle_state(metsr, vid, private_veh=private_veh)
        if (
            record
            and record.get("x") is not None
            and record.get("y") is not None
            and int(record.get("state", 1)) > 0
            and vehicle_has_road(record)
        ):
            return record
        metsr.tick()
    raise RuntimeError(f"Vehicle {vid} did not enter the road network within {max_ticks} ticks.")


def start_metsr_run(config):
    sim_dirs = prepare_sim_dirs(config)
    run_simulation_in_docker(config)
    port = METSR_PORT
    if port is None:
        port = config.ports[0] if hasattr(config, "ports") else config.metsr_port[0]
    client = METSRClient(
        host=METSR_HOST,
        port=int(port),
        sim_folder=sim_dirs[0] if sim_dirs else None
    )
    return client, sim_dirs


def stop_metsr_run(client):
    if client is None:
        return
    try:
        client.terminate()
    except Exception:
        client.close()


def spawn_ego_vehicle(metsr):
    metsr.tick(10)
    metsr.generate_trip_between_roads(EGO_VID, ORIGIN_ROAD, DESTINATION_ROAD)
    metsr.update_vehicle_sensor_type(EGO_VID, "cv2x", private_veh=PRIVATE_VEHICLE)
    return wait_until_on_road(metsr, EGO_VID)

In [ ]:
def v2x_tick(veins, tick, vehicles, messages, phase):
    mobility = build_mobility_records(vehicles)
    # sync_tick returns a dict with bridge fields and a "communication_records" list with one record per message-vehicle pair.
    # mobility's format is a list of dicts with keys like "id", "x", "y", "speed", etc. for each vehicle, which Simu5G/bridge uses to update the mobility of its simulated vehicles.
    result = veins.sync_tick(
        tick=tick,
        vehicles=mobility,
        bsm_messages=messages,
        attacks=[],
        duration_s=0.1,
    )
    implementation = bridge_field(result, "backend_implementation")
    if implementation != "simu5g_cellular_uu":
        raise RuntimeError(
            "The bridge did not report backend_implementation=simu5g_cellular_uu. "
            f"Reported value: {implementation!r}. Start veins_bridge/omnetpp/run_sim5g_uu.sh."
        )
    rows = communication_records_from_result(result, vehicles, messages)
    for row in rows:
        row["phase"] = phase
        row["payload_x"] = row.get("tx_x")
        row["payload_y"] = row.get("tx_y")
        row["payload_speed_mps"] = row.get("tx_speed_mps")
    return result, rows


def show_records(rows, limit=TABLE_ROWS):
    def clean(value, digits=3):
        if value is None or isinstance(value, bool):
            return value
        if isinstance(value, int):
            return value
        if isinstance(value, float):
            return round(value, digits) if math.isfinite(value) else None
        if isinstance(value, dict):
            return {key: clean(item, digits) for key, item in value.items()}
        if isinstance(value, list):
            return [clean(item, digits) for item in value]
        return value

    def short_text(value, max_chars=180):
        if value is None:
            return None
        text = json.dumps(value, sort_keys=True, default=str) if isinstance(value, (dict, list)) else str(value)
        return text if len(text) <= max_chars else text[: max_chars - 3] + "..."

    latencies = [float(row["latency_ms"]) for row in rows if row.get("latency_ms") not in (None, "")]
    distances = [float(row["distance_m"]) for row in rows if row.get("distance_m") not in (None, "")]
    delivered_count = sum(1 for row in rows if row.get("delivered"))
    drop_reasons = {}
    for row in rows:
        if not row.get("delivered"):
            reason = row.get("drop_reason") or "not_delivered"
            drop_reasons[reason] = drop_reasons.get(reason, 0) + 1

    summary = {
        "rows_total": len(rows),
        "rows_shown": min(limit, len(rows)),
        "delivered": delivered_count,
        "dropped": len(rows) - delivered_count,
        "delivery_rate": None if not rows else round(delivered_count / len(rows), 3),
        "selected_by_controller": sum(1 for row in rows if row.get("controller_selected")),
        "latency_ms": None
        if not latencies
        else {
            "mean": clean(sum(latencies) / len(latencies)),
            "min": clean(min(latencies)),
            "max": clean(max(latencies)),
        },
        "distance_m": None
        if not distances
        else {
            "mean": clean(sum(distances) / len(distances)),
            "min": clean(min(distances)),
            "max": clean(max(distances)),
        },
        "drop_reasons": drop_reasons,
    }

    messages = []
    for row in rows[:limit]:
        item = {
            "phase": row.get("phase"),
            "tick": row.get("tick"),
            "message": {
                "id": row.get("message_id"),
                "name": row.get("message_name"),
                "standard": row.get("message_standard"),
                "count": row.get("message_count"),
                "attacked": row.get("attacked"),
                "content": short_text(row.get("message_content")),
            },
            "link": {
                "origin_vehicle_id": row.get("origin_vehicle_id"),
                "target_vehicle_id": row.get("target_vehicle_id"),
                "distance_m": clean(row.get("distance_m")),
                "delivered": row.get("delivered"),
                "latency_ms": clean(row.get("latency_ms")),
                "drop_reason": row.get("drop_reason"),
            },
            "origin_location": {
                "x_m": clean(row.get("origin_x")),
                "y_m": clean(row.get("origin_y")),
            },
            "target_location": {
                "x_m": clean(row.get("target_x")),
                "y_m": clean(row.get("target_y")),
            },
            "payload_vehicle_state": {
                "x_m": clean(row.get("payload_x")),
                "y_m": clean(row.get("payload_y")),
                "speed_mps": clean(row.get("payload_speed_mps")),
            },
        }
        if row.get("controller_selected") is not None or row.get("command_reason") is not None:
            item["controller"] = {
                "selected": row.get("controller_selected"),
                "command_acc_mps2": clean(row.get("command_acc_mps2")),
                "reason": row.get("command_reason"),
            }
        messages.append(item)

    print(json.dumps({"summary": summary, "messages": messages}, indent=2, default=str))


def trace_rows(trace, phase):
    rows = []
    for segment_tick, row in enumerate(trace):
        item = dict(row)
        item["phase"] = phase
        item["segment_tick"] = segment_tick
        rows.append(item)
    return rows

def numeric_trace_values(trace, key):
    values = []
    for row in trace:
        value = row.get(key)
        values.append(math.nan if value in (None, "") else float(value))
    return values

## Connect to METS-R and the Simu5G bridge

In [ ]:
config = read_run_config(str(RUN_CONFIG_PATH))
metsr, sim_dirs = start_metsr_run(config)

veins = VeinsClient(
    host=VEINS_HOST,
    port=VEINS_PORT,
    connect_timeout=30,
    request_timeout=60,
)
veins.connect()

print(f"connected_to_metsr host={metsr.host} port={metsr.port}")
print(f"connected_to_veins_bridge host={veins.host} port={veins.port}")
print(json.dumps(veins.bridge_info, indent=2))

## Phase 1: record BSM payloads from the live simulator

This phase queries METS-R vehicle states each tick, sends truthful BSMs to Simu5G, and keeps source-to-ego payloads for later replay.

In [ ]:
metsr.generate_trip_between_roads(REPLAY_SOURCE_VID, ORIGIN_ROAD, DESTINATION_ROAD)
metsr.update_vehicle_sensor_type(REPLAY_SOURCE_VID, "cv2x", private_veh=PRIVATE_VEHICLE)
source_state = wait_until_on_road(metsr, REPLAY_SOURCE_VID)
replay_buffer = deque(maxlen=RECORD_TICKS)
record_rows = []

for local_tick in range(RECORD_TICKS):
    source_state = query_vehicle_state(metsr, REPLAY_SOURCE_VID)
    if float(source_state.get("speed", 0.0) or 0.0) > SLOW_SOURCE_TARGET_SPEED_MPS:
        metsr.control_vehicle(REPLAY_SOURCE_VID, SLOW_SOURCE_BRAKE_MPS2, private_veh=PRIVATE_VEHICLE)

    metsr.tick()
    source_state = query_vehicle_state(metsr, REPLAY_SOURCE_VID)

    source = normalize_vehicle(source_state, role="replay_source")
    source["ID"] = REPLAY_SOURCE_RADIO_ID
    source["metsr_vehicle_id"] = REPLAY_SOURCE_VID
    recorder = dict(source)
    recorder.update(
        {
            "ID": EGO_VID,
            "role": "recording_sink",
            "x": float(source.get("x", 0.0)) + 10.0,
            "speed": 0.0,
        }
    )
    participants = [source, recorder]

    source_id = vehicle_id(source)
    recorder_id = vehicle_id(recorder)
    messages = [
        {
            "message_id": f"record:{local_tick}:{source_id}>{recorder_id}:1",
            "tick": local_tick,
            "vehicle_id": source_id,
            "sender_id": source_id,
            "receiver_id": recorder_id,
            "target_vehicle_id": recorder_id,
            "message_name": "BasicSafetyMessage",
            "message_standard": "SAE J2735-aligned",
            "message_count": (local_tick * 16 + 1) % 128,
            "payload_bytes": PAYLOAD_BYTES,
            "tx_time_s": local_tick * float(getattr(config, "sim_step_size", 0.1)),
            "radio_mode": "cv2x",
            "content": (
                f"record BSM tick={local_tick} metsr_veh={REPLAY_SOURCE_VID} radio_veh={source_id} "
                f"pos=({float(source.get('x', 0.0)):.2f},{float(source.get('y', 0.0)):.2f},{float(source.get('z', 0.0)):.2f}) "
                f"speed={float(source.get('speed', 0.0) or 0.0):.2f}mps "
                f"heading={float(source.get('bearing', 0.0) or 0.0):.1f}deg"
            ),
            "map_name": getattr(config, "carla_map", "Town07"),
            "sender_role": source.get("role"),
            "receiver_role": recorder.get("role"),
            "x": source.get("x"),
            "y": source.get("y"),
            "z": source.get("z", 0.0),
            "speed_mps": source.get("speed", 0.0),
            "heading_deg": source.get("bearing", 0.0),
            "truth_x": source.get("x"),
            "truth_y": source.get("y"),
            "truth_speed_mps": source.get("speed", 0.0),
            "truth_heading_deg": source.get("bearing", 0.0),
            "attacked": False,
            "attack_id": "",
            "attack_type": "",
        }
    ]

    _, rows = v2x_tick(veins, local_tick, participants, messages, phase="record")
    replay_buffer.append(copy.deepcopy(messages[0]))
    record_rows.extend(rows)

print(f"recorded_payloads={len(replay_buffer)} communication_rows={len(record_rows)}")
show_records(record_rows, limit=10)

## Phase 2: replay stale BSMs through Simu5G and control the ego vehicle

This starts a fresh METS-R run, generates only the ego vehicle, then injects historical replay-source BSMs through Simu5G. The ego controller reacts only to replay messages that Simu5G reports as delivered.

In [ ]:
if not replay_buffer:
    raise RuntimeError("Run the record phase before the attack phase.")

metsr.reset()
ego_state = spawn_ego_vehicle(metsr)

attack_rows = []
control_trace = []
replay_payloads = list(replay_buffer)

for local_tick in range(ATTACK_TICKS):
    tick = RECORD_TICKS + local_tick
    metsr.tick()
    ego_state = query_vehicle_state(metsr, EGO_VID)
    ego = normalize_vehicle(ego_state, role="ego")
    participants = [ego]
    messages = []
    injected = 0

    replay_index = local_tick + REPLAY_OFFSET_TICKS
    if replay_index < len(replay_payloads):
        stale = replay_payloads[replay_index]
        replay_x = stale.get("x")
        replay_y = stale.get("y")
        replay_z = stale.get("z", 0.0)
        replay_speed = stale.get("speed_mps", stale.get("speed", 0.0))
        replay_heading = stale.get("heading_deg", stale.get("heading", 0.0))

        replay_sender = {
            "ID": REPLAY_SOURCE_RADIO_ID,
            "role": "replayed_source",
            "x": replay_x,
            "y": replay_y,
            "z": replay_z,
            "speed": replay_speed,
            "bearing": replay_heading,
            "private_veh": bool(stale.get("private_veh", PRIVATE_VEHICLE)),
            "sensor_type": "cv2x",
            "map_name": stale.get("map_name", getattr(config, "carla_map", "Town07")),
        }
        replay_bsm = copy.deepcopy(stale)
        original_id = replay_bsm.get("message_id", "recorded")
        replay_bsm.update(
            {
                "message_id": f"replay:{tick}:{original_id}:1",
                "tick": tick,
                "vehicle_id": REPLAY_SOURCE_RADIO_ID,
                "sender_id": REPLAY_SOURCE_RADIO_ID,
                "receiver_id": vehicle_id(ego),
                "target_vehicle_id": vehicle_id(ego),
                "message_count": (tick * 16 + 1) % 128,
                "tx_time_s": tick * float(getattr(config, "sim_step_size", 0.1)),
                "x": replay_x,
                "y": replay_y,
                "z": replay_z,
                "speed_mps": replay_speed,
                "speed": replay_speed,
                "heading_deg": replay_heading,
                "heading": replay_heading,
                "attacked": True,
                "attack_id": "simu5g_bsm_replay",
                "attack_type": "bsm_replay"
            }
        )
        participants.append(replay_sender)
        messages.append(replay_bsm)
        injected = 1

    _, rows = v2x_tick(veins, tick, participants, messages, phase="attack")

    ego_x = float(ego.get("x", 0.0))
    ego_y = float(ego.get("y", 0.0))
    ego_speed = float(ego.get("speed", 0.0) or 0.0)
    front_candidates = []
    for row in rows:
        if not row.get("delivered") or row.get("target_vehicle_id") != EGO_VID:
            continue
        if row.get("payload_x") is None or row.get("payload_y") is None:
            continue
        dx = float(row["payload_x"]) - ego_x
        dy = float(row["payload_y"]) - ego_y
        distance_m = math.hypot(dx, dy)
        front_speed = float(row.get("payload_speed_mps") or 0.0)
        front_candidates.append((distance_m, front_speed, row))

    if front_candidates:
        front_distance, front_speed, chosen = min(front_candidates, key=lambda item: item[0])
        relative_speed = ego_speed - front_speed
        if front_distance < MIN_GAP_M:
            command_acc = -MAX_ACCELERATION_MPS2
            command_reason = "replay_front_inside_min_gap"
        elif front_distance < 2 * MIN_GAP_M and relative_speed > 0:
            command_acc = -min(MAX_ACCELERATION_MPS2, relative_speed ** 2 / (2 * max(front_distance - MIN_GAP_M, 0.1)))
            command_reason = "replay_front_closing"
        else:
            command_acc = KP_SPEED * (TARGET_VELOCITY_MPS - ego_speed)
            command_reason = "replay_delivered_but_not_braking"
    else:
        command_acc = KP_SPEED * (TARGET_VELOCITY_MPS - ego_speed)
        chosen = None
        front_distance = None
        front_speed = None
        command_reason = "no_delivered_replay_fallback_speed_control"
    command_acc = max(-MAX_ACCELERATION_MPS2, min(MAX_ACCELERATION_MPS2, command_acc))
    observed_acc = ego_state.get("acc", command_acc)
    if vehicle_has_road(ego_state):
        metsr.control_vehicle(EGO_VID, command_acc, private_veh=PRIVATE_VEHICLE)

    chosen_message_id = None if chosen is None else chosen.get("message_id")
    for row in rows:
        row["command_acc_mps2"] = command_acc
        row["ego_acc_mps2"] = observed_acc
        row["controller_selected"] = row.get("message_id") == chosen_message_id
        row["ego_speed_mps"] = ego.get("speed")
        row["command_reason"] = command_reason
        row["selected_distance_m"] = front_distance
    attack_rows.extend(rows)
    control_trace.append(
        {
            "tick": tick,
            "ego_speed_mps": ego.get("speed"),
            "ego_acc_mps2": observed_acc,
            "command_acc_mps2": command_acc,
            "selected_message_id": chosen_message_id,
            "selected_attack_type": None if chosen is None else chosen.get("attack_type"),
            "selected_distance_m": front_distance,
            "selected_front_speed_mps": front_speed,
            "front_candidate_count": len(front_candidates),
            "command_reason": command_reason,
            "delivered_messages_to_ego": sum(1 for row in rows if row.get("delivered") and row.get("target_vehicle_id") == EGO_VID),
            "injected_bsm_messages": injected,
        }
    )

print(f"attack_rows={len(attack_rows)}")
show_records(attack_rows, limit=TABLE_ROWS)

## Phase 3: normal ego control

This starts another fresh METS-R run and applies the original Attack 1 normal controller: the ego vehicle tries to reach the target speed and does not react to front-vehicle BSM candidates.

In [ ]:
metsr.reset()
ego_state = spawn_ego_vehicle(metsr)

normal_rows = []
normal_trace = []

for local_tick in range(NORMAL_TICKS):
    tick = RECORD_TICKS + ATTACK_TICKS + local_tick
    metsr.tick(1, wait_forever=True)
    ego_state = query_vehicle_state(metsr, EGO_VID)
    ego = normalize_vehicle(ego_state, role="ego")

    _, rows = v2x_tick(veins, tick, [ego], [], phase="normal")
    ego_speed = float(ego.get("speed", 0.0) or 0.0)
    command_acc = KP_SPEED * (TARGET_VELOCITY_MPS - ego_speed)
    command_acc = max(-MAX_ACCELERATION_MPS2, min(MAX_ACCELERATION_MPS2, command_acc))
    observed_acc = ego_state.get("acc", command_acc)
    if vehicle_has_road(ego_state):
        metsr.control_vehicle(EGO_VID, command_acc, private_veh=PRIVATE_VEHICLE)

    for row in rows:
        row["command_acc_mps2"] = command_acc
        row["ego_acc_mps2"] = observed_acc
        row["ego_speed_mps"] = ego.get("speed")
    normal_rows.extend(rows)
    normal_trace.append(
        {
            "tick": tick,
            "ego_speed_mps": ego.get("speed"),
            "ego_acc_mps2": observed_acc,
            "command_acc_mps2": command_acc,
            "selected_message_id": None,
            "selected_attack_type": None,
            "selected_distance_m": None,
            "selected_front_speed_mps": None,
            "front_candidate_count": 0,
            "command_reason": "normal_speed_control",
            "delivered_messages_to_ego": 0,
        }
    )

print(f"normal_rows={len(normal_rows)}")
show_records(normal_rows, limit=TABLE_ROWS)

## Summarize and visualize the run

In [ ]:
all_rows = record_rows + attack_rows + normal_rows
delivered = [row for row in all_rows if row.get("delivered")]
attack_delivered = [row for row in attack_rows if row.get("delivered") and row.get("attacked")]
latencies = [float(row["latency_ms"]) for row in delivered if row.get("latency_ms") not in (None, "")]

normal_speed = numeric_trace_values(normal_trace, "ego_speed_mps")
attacked_speed = numeric_trace_values(control_trace, "ego_speed_mps")
normal_acc = numeric_trace_values(normal_trace, "ego_acc_mps2")
attacked_acc = numeric_trace_values(control_trace, "ego_acc_mps2")

min_len = min(len(normal_speed), len(attacked_speed))
if np is not None and min_len:
    speed_delta = np.abs(
        np.asarray(attacked_speed[:min_len], dtype=float)
        - np.asarray(normal_speed[:min_len], dtype=float)
    )
else:
    speed_delta = [abs(attacked_speed[i] - normal_speed[i]) for i in range(min_len)]

recovery_tick = None
recovery_window = 100
recovery_threshold = 0.5
for index in range(RECORD_TICKS, max(RECORD_TICKS, min_len - recovery_window)):
    window = speed_delta[index : index + recovery_window]
    if len(window) and max(window) <= recovery_threshold:
        recovery_tick = index
        break

sim_step = float(getattr(config, "sim_step_size", 0.1))
bsm_replay_result = {
    "recorded_bsm_messages": int(sum(1 for row in record_rows if row.get("origin_vehicle_id") == REPLAY_SOURCE_RADIO_ID)),
    "ticks_with_replay_data": len(replay_buffer),
    "injected_bsm_messages": int(sum(row.get("injected_bsm_messages", 0) for row in control_trace)),
    "replay_duration_seconds": RECORD_TICKS * sim_step,
    "recovery_start_seconds": None if recovery_tick is None else recovery_tick * sim_step,
    "normal_speed": normal_speed,
    "attacked_speed": attacked_speed,
    "normal_acc": normal_acc,
    "attacked_acc": attacked_acc,
}
attack_reason_counts = {}
for row in control_trace:
    reason = row.get("command_reason", "unknown")
    attack_reason_counts[reason] = attack_reason_counts.get(reason, 0) + 1

summary = {
    "backend_implementation": bridge_field(veins.bridge_info, "backend_implementation", "see sync_tick rows"),
    "record_rows": len(record_rows),
    "normal_rows": len(normal_rows),
    "attack_rows": len(attack_rows),
    "delivered_rows": len(delivered),
    "delivered_replay_rows": len(attack_delivered),
    "mean_latency_ms": None if not latencies else sum(latencies) / len(latencies),
    "max_latency_ms": None if not latencies else max(latencies),
    "recorded_bsm_messages": bsm_replay_result["recorded_bsm_messages"],
    "ticks_with_replay_data": bsm_replay_result["ticks_with_replay_data"],
    "injected_bsm_messages": bsm_replay_result["injected_bsm_messages"],
    "replay_duration_seconds": bsm_replay_result["replay_duration_seconds"],
    "recovery_start_seconds": bsm_replay_result["recovery_start_seconds"],
    "attack_ticks_with_delivered_replay": int(sum(1 for row in control_trace if row.get("front_candidate_count", 0) > 0)),
    "attack_ticks_braking_from_replay": int(sum(1 for row in control_trace if str(row.get("command_reason", "")).startswith("replay_front"))),
    "attack_command_reasons": attack_reason_counts,
}
print(json.dumps(summary, indent=2))

comparison_trace = trace_rows(control_trace, "attack") + trace_rows(normal_trace, "normal")
trace_preview = [
    {
        "phase": row.get("phase"),
        "segment_tick": row.get("segment_tick"),
        "tick": row.get("tick"),
        "ego_speed_mps": row.get("ego_speed_mps"),
        "ego_acc_mps2": row.get("ego_acc_mps2"),
        "selected_message_id": row.get("selected_message_id"),
        "selected_distance_m": row.get("selected_distance_m"),
        "command_reason": row.get("command_reason"),
        "delivered_messages_to_ego": row.get("delivered_messages_to_ego"),
    }
    for row in comparison_trace[: 2 * TABLE_ROWS]
]
print(
    json.dumps(
        {
            "trace_rows_total": len(comparison_trace),
            "trace_rows_shown": len(trace_preview),
            "trace_preview": trace_preview,
        },
        indent=2,
        default=str,
    )
)

if plt is not None:
    if np is not None:
        x_seconds = np.arange(len(normal_speed)) * sim_step
    else:
        x_seconds = [index * sim_step for index in range(len(normal_speed))]

    fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
    axes[0, 0].plot(x_seconds, normal_speed, lw=0.6)
    axes[0, 1].plot(x_seconds, attacked_speed, lw=0.6, color="tab:red")
    axes[1, 0].plot(x_seconds, normal_acc, lw=0.6)
    axes[1, 1].plot(x_seconds, attacked_acc, lw=0.6, color="tab:red")

    axes[0, 0].set_title("Normal speed")
    axes[0, 1].set_title("Attacked speed")
    axes[1, 0].set_title("Normal acceleration")
    axes[1, 1].set_title("Attacked acceleration")
    axes[0, 0].set_ylabel("Speed (m/s)")
    axes[1, 0].set_ylabel("Acceleration (m/s^2)")
    axes[1, 0].set_xlabel("Second")
    axes[1, 1].set_xlabel("Second")

    for axis in axes.ravel():
        axis.axvline(bsm_replay_result["replay_duration_seconds"], color="0.4", ls="--", lw=0.8)
        if bsm_replay_result["recovery_start_seconds"] is not None:
            axis.axvline(bsm_replay_result["recovery_start_seconds"], color="tab:green", ls=":", lw=1.0)

    plt.suptitle("BSM replay attack: normal vs replayed historical stream")
    fig.tight_layout()
    plt.show()
else:
    print("matplotlib is not installed; skipped speed/acceleration plots.")

{key: bsm_replay_result[key] for key in ("recorded_bsm_messages", "ticks_with_replay_data", "injected_bsm_messages", "replay_duration_seconds", "recovery_start_seconds")} if bsm_replay_result else None

## Cleanup

Closing the clients leaves an externally-started METS-R simulator running. If this notebook launched METS-R and `TERMINATE_METSR_WHEN_DONE` is true, it sends `CTRL_end`.

In [ ]:
try:
    veins.close()
except NameError:
    pass

try:
    stop_metsr_run(metsr)
except NameError:
    pass